# NLP Feature Setup: 10-K AI-disclosure front end

Shared front end for the three AI-maturity dimensions (strategy,
operations, governance). It resolves one 10-K per firm of the active universe (`UNIVERSE`,
Fortune 500 or S&P 500; pinned to a target fiscal year, `TARGET_FISCAL_YEAR`), extracts Items 1, 1A, and 7, segments and AI-filters their
sentences, and validates the AI keyword dictionary. The cached outputs
(`filings`, `sentences`, `sentence_totals`, `keyword_distribution`) are
consumed by the per-dimension notebooks, which handle tone scoring and
firm-level aggregation.

Run this notebook once before `strategy.ipynb`, `operations.ipynb`, or
`governance.ipynb`. It is the only notebook that runs the spaCy pass.

**Prerequisites.**
- The conda environment from `environment.yml` is active (provides
  `spacy` + `en_core_web_sm`, `edgartools`, `scikit-learn`).
- `EDGAR_IDENTITY` set to "Your Name email@host" (the setup cell falls
  back to a default if unset).

In [1]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import logging
import os
import random
import sys
import webbrowser
from collections import Counter
from pathlib import Path

import pandas as pd
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.universe import UNIVERSES, load_universe
from src.indicators.common.io import (
    cache_path,
    clear_cache,
    load_cached_step,
    save_cached_step,
)
from src.indicators.nlp_features import (
    AI_KEYWORDS,
    assemble_sections,
    build_ai_sentence_review,
    extract_items,
    fetch_filing,
    filter_ai_sentences,
    resolve_filings,
)
from src.indicators.nlp_features.filter import (
    aggregate_keyword_counts,
    count_keyword_occurrences,
    is_ai_sentence,
    split_sentences,
)
from src.indicators.nlp_features.aggregate import ALL_ITEMS, MIN_ITEM_SENTENCES_FOR_PARSE

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

os.environ.setdefault("EDGAR_IDENTITY", "Timo Koba kab.timo3@gmail.com")

# Cached steps under data_cache/indicators/nlp_features/. Only `edgar_sections`
# and `raw_text` (section 2a) touch EDGAR; everything else is rebuilt offline
# from them, so re-tuning the parser or splitter never re-downloads:
#   filings          - resolved 10-K per ticker (section 1)
#   edgar_sections   - edgartools structured Item picks    (2a, EDGAR)
#   raw_text         - full filing text for the regex path (2a, EDGAR)
#   sections         - final Items + parser provenance     (2b, local)
#   sentences / sentence_totals / keyword_distribution      (2b, local)
# Toggle FORCE_REFRESH to recompute all, or clear_cache("nlp_features", UNIVERSE, "<step>").
FORCE_REFRESH = False
SHARED = "nlp_features"

# Firm universe this run builds ("fortune500" or "sp500"). Every cache,
# indicator parquet, and validation artifact is scoped by this value, so
# both universes coexist side by side. The sentence-level FinBERT caches
# are shared: overlapping firms cost nothing to re-score.
UNIVERSE = "sp500"

# Pin every firm to the 10-K that reports on this fiscal year, so firms with
# different fiscal-year ends stay comparable in the cross-section (a Jan-ending
# filer's FY2025 vs. a Dec-ending filer's FY2025) instead of mixing whatever each
# firm most recently filed. Set to None to take each firm's latest 10-K instead.
TARGET_FISCAL_YEAR = 2025

# Validation-sample knobs (section 3). Surfaced here so a reviewer can find
# them without scrolling into the validation cells.
VALIDATION_DIR = PROJECT_ROOT / "data_clean" / "validation" / UNIVERSE / SHARED
# The keyword-dictionary validation (section 3) measures the dictionary,
# not a universe: it was annotated once on the fortune500 sample and is
# reused as-is for every universe.
_DICT_VALIDATION_DIR = PROJECT_ROOT / "data_clean" / "validation" / "fortune500" / SHARED
ANNOTATION_FILE = _DICT_VALIDATION_DIR / "sample_to_annotate.csv"
KEY_FILE = _DICT_VALIDATION_DIR / "_key.parquet"
SAMPLE_FILINGS_N = 30
POS_N = 100
NEG_N = 100
SEED = 42

## 1. Resolve filings

Look up the 10-K reporting on `TARGET_FISCAL_YEAR` for each firm of the active universe (via CIK where available, ticker otherwise). Foreign filers
(20-F) and resolution failures are dropped and logged.

In [2]:
filings = None if FORCE_REFRESH else load_cached_step(SHARED, "filings", UNIVERSE)
if filings is None:
    filings = resolve_filings(load_universe(UNIVERSE), fiscal_year=TARGET_FISCAL_YEAR)
    save_cached_step(filings, SHARED, "filings", UNIVERSE)
    print(f"Resolved {len(filings)} 10-K filings (saved to {cache_path(SHARED, 'filings', UNIVERSE)})")
else:
    print(f"Loaded {len(filings)} 10-K filings from cache ({cache_path(SHARED, 'filings', UNIVERSE)})")
filings.head()

2026-07-09 22:32:56,501 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]


2026-07-09 22:42:08,790 WARNING edgar.core Subheader 'COMPANY DATA' not found in header ']		IRS NUMBER'


2026-07-09 22:42:08,791 WARNING edgar.core Subheader 'COMPANY DATA' not found in header ']		IRS NUMBER'


2026-07-09 22:43:03,175 WARNING edgar.core Subheader 'COMPANY DATA' not found in header ']		IRS NUMBER'


2026-07-09 22:43:03,176 WARNING edgar.core Subheader 'COMPANY DATA' not found in header ']		IRS NUMBER'


...

2026-07-10 00:25:06,557 WARNING src.indicators.nlp_features.edgar   SNA: no 10-K reporting on fiscal year 2025


2026-07-10 00:25:06,557 WARNING src.indicators.nlp_features.edgar   SWK: no 10-K reporting on fiscal year 2025


2026-07-10 00:25:06,558 WARNING src.indicators.nlp_features.edgar   TXT: no 10-K reporting on fiscal year 2025


Resolved 494 10-K filings (saved to D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_cache\indicators\sp500\nlp_features\filings.parquet)


,cik,ticker,company_name,normalized_company_name,accession_number,fiscal_year,filing_date,form
0,0000066740,MMM,3M Company,3m,0000066740-26-000014,2025,2026-02-03,10-K
1,0000091142,AOS,A. O. Smith Corporation,a o smith,0000091142-26-000008,2025,2026-02-10,10-K
2,0001037868,AME,"AMETEK, Inc.",ametek,0001037868-26-000016,2025,2026-02-17,10-K
3,0001841666,APA,APA Corporation,apa,0001841666-26-000015,2025,2026-02-26,10-K
4,0000732717,T,AT&T Inc.,att,0000732717-26-000120,2025,2026-02-09,10-K


## 2. Parse Item sections + filter AI sentences

Two stages so the EDGAR round-trip happens once and everything else stays local:

- **2a — Download (EDGAR).** For each filing, fetch edgartools' structured Item
  picks and, only when an Item is missing, the full raw text. Cached as
  `edgar_sections` and `raw_text`.
- **2b — Assemble + filter (offline).** Combine the edgartools picks with a
  line-anchored **regex fallback** over the raw text for any missing Item; every
  candidate (structured or regex) passes the same validation, and each returned
  Item is tagged in `sections` with the parser that produced it
  (`edgartools` / `regex`). Then sentence-segment and split each Item's sentences
  into AI-relevant (matched against `AI_KEYWORDS`) plus total cleaned counts. The
  cleaned-sentence totals are the denominator of `ai_sentence_share` in each
  dimension notebook (length-normalized AI-disclosure intensity, Loughran-McDonald
  2011). `sentences` keeps an `item` column so each dimension subsets to its Item.

Re-tuning the regex or the splitter means deleting the `sections` / `sentences` /
`sentence_totals` / `keyword_distribution` steps and re-running 2b — no EDGAR.

**Parse coverage and the `parse_complete` flag.** Regex-sourced sections are
lower-confidence (they can truncate or over-capture), so they are flagged red in
the section-4 review for manual audit. A residual set of large financial firms
(e.g. JPMorgan, MetLife, KKR) cannot be recovered from the 10-K at all because
they *incorporate Items 7, 7A, and 8 by reference from Exhibit 13*, so their MD&A
is not in the primary document; regex cannot help there either. A filing counts
as parsed for an Item only when that Item yields at least
`MIN_ITEM_SENTENCES_FOR_PARSE` clean sentences; `aggregate.py` marks unparsed Items per dimension
(`item_parsed = 0`, NaN share/tone) and writes an identical `parse_complete`
flag into every dimension's output — missing data is marked, not dropped,
and handled at index-composition time.

**Universe reuse.** Filings whose accession number already sits in another universe's `edgar_sections` / `raw_text` caches are copied over instead of re-downloaded, so only genuinely new firms hit EDGAR.

In [3]:
# 2a. Download & cache each filing's edgartools sections + full raw text.
# This is the ONLY EDGAR-bound step. Before fetching, the caches of the other
# universes are reused: identical accession numbers mean identical filings, so
# every overlapping firm is copied over instead of re-downloaded.
print(f"AI_KEYWORDS: {len(AI_KEYWORDS)} keywords")

edgar_sections_df = None if FORCE_REFRESH else load_cached_step(SHARED, "edgar_sections", UNIVERSE)
raw_text_df = None if FORCE_REFRESH else load_cached_step(SHARED, "raw_text", UNIVERSE)

if edgar_sections_df is None or raw_text_df is None:
    wanted = set(filings["accession_number"])
    seed_sec: list[pd.DataFrame] = []
    seed_raw: list[pd.DataFrame] = []
    for _other in (u for u in UNIVERSES if u != UNIVERSE):
        _sec = load_cached_step(SHARED, "edgar_sections", _other)
        _raw = load_cached_step(SHARED, "raw_text", _other)
        if _sec is not None:
            seed_sec.append(_sec[_sec["accession_number"].isin(wanted)])
        if _raw is not None:
            seed_raw.append(_raw[_raw["accession_number"].isin(wanted)])
    seeded_sections = pd.concat(seed_sec, ignore_index=True) if seed_sec else pd.DataFrame()
    seeded_raw = pd.concat(seed_raw, ignore_index=True) if seed_raw else pd.DataFrame()
    have: set = set()
    if len(seeded_sections):
        have |= set(seeded_sections["accession_number"])
    if len(seeded_raw):
        have |= set(seeded_raw["accession_number"])
    to_fetch = filings[~filings["accession_number"].isin(have)]
    print(f"Reusing {len(have & wanted)} filings from other universes; fetching {len(to_fetch)} from EDGAR")

    sec_rows: list[dict] = []
    raw_rows: list[dict] = []
    fetch_errors: list[tuple[str, str]] = []
    for _, row in to_fetch.iterrows():
        acc = row["accession_number"]
        try:
            fetched = fetch_filing(acc)
            for item, text in fetched.edgar_sections.items():
                sec_rows.append({"accession_number": acc, "cik": row["cik"], "item": item, "text": text})
            if fetched.raw_text:
                raw_rows.append({"accession_number": acc, "cik": row["cik"], "text": fetched.raw_text})
        except Exception as exc:
            fetch_errors.append((acc, str(exc)))

    edgar_sections_df = pd.concat([seeded_sections, pd.DataFrame(sec_rows)], ignore_index=True)
    raw_text_df = pd.concat([seeded_raw, pd.DataFrame(raw_rows)], ignore_index=True)
    save_cached_step(edgar_sections_df, SHARED, "edgar_sections", UNIVERSE)
    save_cached_step(raw_text_df, SHARED, "raw_text", UNIVERSE)
    print(f"Cached edgartools sections for {edgar_sections_df['accession_number'].nunique()} filings "
          f"({len(edgar_sections_df)} Item rows) and raw text for {raw_text_df['accession_number'].nunique()} filings. "
          f"Fetch errors: {len(fetch_errors)}")
else:
    print(f"Loaded cached edgartools sections ({len(edgar_sections_df)} Item rows) and raw text "
          f"({len(raw_text_df)} filings) — no EDGAR calls.")

AI_KEYWORDS: 80 keywords


2026-07-10 00:25:07,843 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]


Reusing 316 filings from other universes; fetching 178 from EDGAR


2026-07-10 00:25:11,792 INFO edgar.documents.extractors.toc_section_detector TOC detection found 21 sections


2026-07-10 00:25:11,793 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 21 sections found


...

2026-07-10 00:47:07,615 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 23 sections found


2026-07-10 00:47:09,115 INFO edgar.core Identity of the Edgar REST client set to [Timo Koba kab.timo3@gmail.com]


2026-07-10 00:47:12,690 INFO edgar.documents.extractors.toc_section_detector TOC detection found 24 sections


2026-07-10 00:47:12,691 INFO edgar.documents.extractors.hybrid_section_detector TOC detection successful: 24 sections found


Cached edgartools sections for 494 filings (1379 Item rows) and raw text for 494 filings. Fetch errors: 0


In [4]:
# 2b. Assemble final sections (edgartools + validated regex fallback, with
# per-Item provenance), then segment + AI-filter. Fully offline: rebuilt from the
# 2a caches, so this is what you re-run after changing parse.py or filter.py.
sections_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sections", UNIVERSE)
sentences_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sentences", UNIVERSE)
sentence_totals_df = None if FORCE_REFRESH else load_cached_step(SHARED, "sentence_totals", UNIVERSE)
keyword_distribution_df = None if FORCE_REFRESH else load_cached_step(SHARED, "keyword_distribution", UNIVERSE)

if any(x is None for x in (sections_df, sentences_df, sentence_totals_df, keyword_distribution_df)):
    edgar_by_acc = {
        acc: dict(zip(g["item"], g["text"])) for acc, g in edgar_sections_df.groupby("accession_number")
    }
    raw_by_acc = dict(zip(raw_text_df["accession_number"], raw_text_df["text"])) if len(raw_text_df) else {}

    sec_rows: list[dict] = []
    all_sentences: list[pd.DataFrame] = []
    totals_rows: list[dict] = []
    for _, row in filings.iterrows():
        acc = row["accession_number"]
        sections, sources = assemble_sections(edgar_by_acc.get(acc, {}), raw_by_acc.get(acc, ""))
        for item, text in sections.items():
            sec_rows.append(
                {"accession_number": acc, "cik": row["cik"], "item": item, "source": sources[item], "text": text}
            )
        sub, totals = filter_ai_sentences(sections, accession_number=acc, cik=row["cik"])
        if len(sub) > 0:
            all_sentences.append(sub)
        totals_rows.append(
            {
                "accession_number": acc,
                "cik": row["cik"],
                "n_sentences_item_1": int(totals.get("item_1", 0)),
                "n_sentences_item_1a": int(totals.get("item_1a", 0)),
                "n_sentences_item_7": int(totals.get("item_7", 0)),
            }
        )

    sections_df = pd.DataFrame(sec_rows)
    save_cached_step(sections_df, SHARED, "sections", UNIVERSE)

    sentences_df = pd.concat(all_sentences, ignore_index=True) if all_sentences else pd.DataFrame()
    save_cached_step(sentences_df, SHARED, "sentences", UNIVERSE)

    sentence_totals_df = pd.DataFrame(totals_rows)
    save_cached_step(sentence_totals_df, SHARED, "sentence_totals", UNIVERSE)

    # Concept frequencies over the analyzed AI sentences (every keyword occurrence
    # lies inside an AI sentence, so this is exact and needs no separate pass).
    kw_counter: Counter = Counter()
    for _sent in sentences_df["sentence"]:
        kw_counter.update(count_keyword_occurrences(_sent))
    keyword_distribution_df = aggregate_keyword_counts(kw_counter)
    save_cached_step(keyword_distribution_df, SHARED, "keyword_distribution", UNIVERSE)
    print(f"Assembled sections for {sections_df['accession_number'].nunique()} filings; "
          f"{len(sentences_df)} AI-relevant sentences.")
else:
    print(f"Loaded {len(sentences_df)} AI sentences from cache ({cache_path(SHARED, 'sentences', UNIVERSE)})")

_by_source = sections_df["source"].value_counts()
print(f"Sections by parser: edgartools={int(_by_source.get('edgartools', 0))}, "
      f"regex={int(_by_source.get('regex', 0))}  "
      f"(regex sections are flagged red in the section-4 review)")
print(f"Filings with at least one AI mention: "
      f"{sentences_df['accession_number'].nunique() if len(sentences_df) else 0}")

print(f"\nPer-Item parse coverage (>= {MIN_ITEM_SENTENCES_FOR_PARSE} clean sentences):")
_n_filings = len(sentence_totals_df)
for _item in ALL_ITEMS:
    _ok = int((sentence_totals_df[f"n_sentences_{_item}"] >= MIN_ITEM_SENTENCES_FOR_PARSE).sum())
    print(f"  {_item:8s}: {_ok:3d}/{_n_filings}")
_complete = (
    sentence_totals_df[[f"n_sentences_{i}" for i in ALL_ITEMS]]
    .ge(MIN_ITEM_SENTENCES_FOR_PARSE)
    .all(axis=1)
)
print(f"  parse_complete (all three Items): {int(_complete.sum())}/{_n_filings}  "
      f"(the {int((~_complete).sum())} incomplete filings are dropped uniformly via parse_complete)")

print("\nAI sentences by Item (extensive-margin raw counts):")
print(sentences_df["item"].value_counts().to_string() if len(sentences_df) else "(none)")
print(f"\nKeyword concept groups: {len(keyword_distribution_df)}  "
      f"(0-count: {int((keyword_distribution_df['count'] == 0).sum())})")
print("Top 15 concept groups by total occurrences:")
print(keyword_distribution_df.head(15).to_string(index=False))

Assembled sections for 494 filings; 8314 AI-relevant sentences.
Sections by parser: edgartools=1362, regex=74  (regex sections are flagged red in the section-4 review)
Filings with at least one AI mention: 472

Per-Item parse coverage (>= 25 clean sentences):
  item_1  : 487/494
  item_1a : 481/494
  item_7  : 457/494
  parse_complete (all three Items): 446/494  (the 48 incomplete filings are dropped uniformly via parse_complete)

AI sentences by Item (extensive-margin raw counts):
item
item_1a    5197
item_1     2495
item_7      622

Keyword concept groups: 58  (0-count: 18)
Top 15 concept groups by total occurrences:
                           keyword  count  share_pct
                                ai   8443      71.44
           artificial intelligence   1612      13.64
                  machine learning    591       5.00
                     generative ai    504       4.26
                           agentic    185       1.57
generative artificial intelligence     92       0.78
  

### Cross-section duplication audit

Regression check for the parser's overlap guards: shares of sentence chunks
appearing in two Items of the same filing. Small shares are genuine
in-document repetition (segment descriptions and forward-looking
disclaimers restated across Items); a share near 1.0 would mean a
mislabeled or over-captured section double-counting every sentence — that
must not happen.

In [ ]:
from src.indicators.nlp_features.parse import duplication_audit

_dup = duplication_audit(sections_df)
_tick = dict(zip(filings["accession_number"], filings["ticker"]))
print(f"section pairs sharing >= 3 sentence chunks: {len(_dup)}")
if len(_dup):
    _worst = _dup.sort_values("pct_of_smaller", ascending=False).head(10).copy()
    _worst.insert(0, "ticker", _worst["accession_number"].map(_tick))
    print(f"max share of smaller section: {_dup['pct_of_smaller'].max():.2f}")
    print(_worst.drop(columns="accession_number").to_string(index=False))
    assert _dup["pct_of_smaller"].max() < 0.6, (
        "a section pair shares most of its text — mislabeled or over-captured section"
    )

## 3. Validation: keyword dictionary precision / recall / F1 *(optional, for appendix)*

Stratified random sample of 200 sentences (100 dict-positive, 100
dict-negative) drawn from a 30-filing subsample. Sentences are shuffled
and the source label is hidden so the annotator labels blind.

**Workflow.**
1. Run the **Build sample** cell once. It writes
   `data_clean/validation/nlp_features/sample_to_annotate.csv`.
2. Open that CSV, fill the `gold` column with `1` if the sentence
   substantively discusses AI/ML technology, deployment, governance, or
   strategy, else `0`. Save (keep the same filename).
3. Run the **Compute metrics** cell. It joins your annotations with the
   hidden source key and prints precision, recall, F1 plus example errors.

To regenerate the sample, delete the CSV.

This validation measures the keyword dictionary, not a universe; the annotated sample lives under `validation/fortune500/` and is reused for every universe.

In [5]:
# Self-load `filings` from cache if it isn't already in memory (so this cell
# can run after a kernel restart without re-executing section 1).
try:
    filings
except NameError:
    filings = load_cached_step(SHARED, "filings", UNIVERSE)
    if filings is None:
        raise RuntimeError("No cached filings found. Run section 1 (Resolve filings) first.")
    print(f"Loaded {len(filings)} filings from cache for validation")

if ANNOTATION_FILE.exists():
    print(f"Sample already exists: {ANNOTATION_FILE}")
    print("Delete the file to regenerate, or annotate it and run the metrics cell.")
else:
    pool_filings = filings.sample(n=min(SAMPLE_FILINGS_N, len(filings)), random_state=SEED)
    pool_pos: list[dict] = []
    pool_neg: list[dict] = []
    for _, row in pool_filings.iterrows():
        try:
            sections = extract_items(row["accession_number"])
        except Exception:
            continue
        for item, text in sections.sections.items():
            for sent in split_sentences(text):
                rec = {"sentence": sent, "item": item, "accession_number": row["accession_number"]}
                (pool_pos if is_ai_sentence(sent) else pool_neg).append(rec)

    rng = random.Random(SEED)
    rng.shuffle(pool_pos)
    rng.shuffle(pool_neg)
    selected = pd.DataFrame(
        [{**r, "source": "pos"} for r in pool_pos[:POS_N]]
        + [{**r, "source": "neg"} for r in pool_neg[:NEG_N]]
    )
    selected = selected.sample(frac=1, random_state=SEED).reset_index(drop=True)
    selected.insert(0, "id", range(len(selected)))

    VALIDATION_DIR.mkdir(parents=True, exist_ok=True)
    selected[["id", "source"]].to_parquet(KEY_FILE, index=False)
    annotation = selected[["id", "sentence", "item", "accession_number"]].copy()
    annotation["gold"] = ""
    annotation.to_csv(ANNOTATION_FILE, index=False, encoding="utf-8-sig")

    print(f"Pool sizes: {len(pool_pos)} positives / {len(pool_neg)} negatives across {len(pool_filings)} filings")
    print(f"Wrote {len(annotation)} sentences to: {ANNOTATION_FILE}")
    print("Open it, fill the 'gold' column with 1 or 0, save, then run the next cell.")

Sample already exists: D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_clean\validation\fortune500\nlp_features\sample_to_annotate.csv
Delete the file to regenerate, or annotate it and run the metrics cell.


In [6]:
if not ANNOTATION_FILE.exists() or not KEY_FILE.exists():
    raise RuntimeError(
        f"Validation files not found under {VALIDATION_DIR}. Run the Build sample cell first."
    )

annotated = pd.read_csv(ANNOTATION_FILE, encoding="utf-8-sig")
key = pd.read_parquet(KEY_FILE)
df = annotated.merge(key, on="id", how="inner")

df["gold"] = pd.to_numeric(df["gold"], errors="coerce")
unannotated = int(df["gold"].isna().sum())
df = df.dropna(subset=["gold"]).copy()
df["gold"] = df["gold"].astype(int)
df["pred"] = (df["source"] == "pos").astype(int)

if unannotated > 0:
    print(f"WARNING: {unannotated} rows have no gold label and were skipped.")

P = precision_score(df["gold"], df["pred"], zero_division=0)
R = recall_score(df["gold"], df["pred"], zero_division=0)
F = f1_score(df["gold"], df["pred"], zero_division=0)
tn, fp, fn, tp = confusion_matrix(df["gold"], df["pred"], labels=[0, 1]).ravel()

print(f"N annotated:  {len(df)}")
print(f"Precision:    {P:.3f}   ({tp} TP / {tp + fp} predicted positives)")
print(f"Recall:       {R:.3f}   ({tp} TP / {tp + fn} actual positives)")
print(f"F1:           {F:.3f}")
print(f"Confusion:    TP={tp}  FP={fp}  TN={tn}  FN={fn}")

print("\n-- Up to 5 false positives (dict said AI, you said not) --")
for _, r in df[(df["pred"] == 1) & (df["gold"] == 0)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")
print("\n-- Up to 5 false negatives (you said AI, dict missed it) --")
for _, r in df[(df["pred"] == 0) & (df["gold"] == 1)].head(5).iterrows():
    print(f"  [{r['item']}] {r['sentence']}")

N annotated:  200
Precision:    1.000   (100 TP / 100 predicted positives)
Recall:       1.000   (100 TP / 100 actual positives)
F1:           1.000
Confusion:    TP=100  FP=0  TN=100  FN=0

-- Up to 5 false positives (dict said AI, you said not) --

-- Up to 5 false negatives (you said AI, dict missed it) --


## 4. Manual parse review (HTML)

Build one self-contained HTML page listing every parse-complete firm and its
AI-relevant sentences per Item, in document order (runs of non-AI sentences are
elided with a count). Each Item is badged with the parser that produced it, and
**regex-sourced Items are flagged red** — review those, since the regex fallback
can truncate or over-capture where edgartools declined to parse. The page opens
automatically in your browser. It reads only the cached frames, so it is safe to
re-run any time (no EDGAR, no recompute).

In [7]:
# Self-load from cache so this runs standalone after a kernel restart.
try:
    filings
except NameError:
    filings = load_cached_step(SHARED, "filings", UNIVERSE)

_sent = load_cached_step(SHARED, "sentences", UNIVERSE)
_tot = load_cached_step(SHARED, "sentence_totals", UNIVERSE)
_sec = load_cached_step(SHARED, "sections", UNIVERSE)
if any(x is None for x in (filings, _sent, _tot, _sec)):
    raise RuntimeError("Run section 2 first — sentences / sentence_totals / sections caches are missing.")

review_path = VALIDATION_DIR / "parse_review_ai_sentences.html"
build_ai_sentence_review(_sent, filings, _tot, _sec, review_path)
_n_regex = int((_sec["source"] == "regex").sum())
print(f"Wrote {review_path}")
print(f"{_n_regex} regex-parsed section(s) flagged red — please review those.")
webbrowser.open(review_path.resolve().as_uri())

Wrote D:\A Studium\MSc Management and Technology\(5) Wintersemester 2025 26\Masterarbeit\Code\data_clean\validation\sp500\nlp_features\parse_review_ai_sentences.html
74 regex-parsed section(s) flagged red — please review those.


True